# Flip Perturbation Attention Analysis — Varied-Context Edition

Data source: **`profiling_logs_v6/`** — flip perturbations only, three injections
(banking/it7, slack/it4, travel/it6), each run against a random sample of user-task
contexts from that suite.  N ∈ {1, 2, 4} tokens flipped, fluency distance unconstrained.

Focus: **flip perturbations only**.  
Key questions:
1. Which contexts does each injection succeed in at baseline?
2. Do attention patterns at the first 10 reasoning tokens distinguish success from failure — consistently across contexts?

In [1]:
import sys
sys.path.append('..')
import sqlite3, json, dill, torch, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from pathlib import Path
from tqdm.auto import tqdm

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Ready.')

Ready.


/workspace/better_opts_attacks/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
LOG_UNC   = Path('profiling_logs_v6')

# One entry per injection — pooling all user-task contexts in analysis
EXAMPLES = [
    ('banking', 'injection_task_7'),
    ('slack',   'injection_task_4'),
    ('travel',  'injection_task_6'),
]
EX_LABELS = ['banking/it7', 'slack/it4', 'travel/it6']

# Semantic tag groups
TAG_GROUPS = {
    'payload':   ['attack_payload', 'attack_prefix', 'attack_suffix'],
    'user':      ['user_instruction'],
    'tool_env':  ['tool_env_data'],
    'developer': ['developer_instructions', 'developer_tools'],
    'system':    ['system_meta'],
    'control':   ['frame_boundary', 'frame_message', 'frame_role',
                  'frame_channel', 'frame_constrain', 'frame_channel_name',
                  'frame_constrain_type', 'frame_metadata'],
    'assistant': ['assistant_reasoning', 'assistant_tool_call',
                  'assistant_commentary', 'assistant_final'],
}
GRP_COLORS = {
    'payload': '#e74c3c', 'user': '#3498db', 'tool_env': '#2ecc71',
    'developer': '#8e44ad', 'system': '#95a5a6', 'control': '#bdc3c7',
    'assistant': '#f39c12',
}
GRP_NAMES = list(TAG_GROUPS.keys())
N_GRPS = len(GRP_NAMES)

LOCAL_LAYERS  = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22]
GLOBAL_LAYERS = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23]

N_LAYERS, N_HEADS, N_STEPS = 24, 64, 10

In [ ]:
def load_log_dir(log_dir, band_label=None):
    """Load run metadata + distances + cap paths from a log directory."""
    log_dir = Path(log_dir)
    conn = sqlite3.connect(log_dir / 'experiment_logs.db')
    
    run_rows = pd.read_sql("SELECT metadata_json FROM logs WHERE event='profiling_run'", conn)
    run_df = pd.DataFrame([json.loads(r) for r in run_rows['metadata_json']])
    
    dist_rows = pd.read_sql(
        "SELECT metadata_json, object_data FROM logs WHERE event='profiling_distances'", conn)
    dist_list = []
    for _, row in dist_rows.iterrows():
        meta = json.loads(row['metadata_json'])
        obj  = dill.loads(row['object_data'])
        dist_list.append({'run_id': meta['run_id'], 'fluency': obj.get('fluency', np.nan)})
    dist_df = pd.DataFrame(dist_list)
    
    cap_rows = pd.read_sql(
        "SELECT metadata_json, object_data FROM logs WHERE event='profiling_capture'", conn)
    cap_list = []
    for _, row in cap_rows.iterrows():
        meta = json.loads(row['metadata_json'])
        obj  = dill.loads(row['object_data'])
        cap_list.append({
            'run_id':   meta['run_id'],
            'cap_path': obj['cap_path'],
            'spans':    obj['spans'],
            'seq_len':  obj['seq_len'],
        })
    cap_df = (pd.DataFrame(cap_list).drop_duplicates(subset=['run_id'], keep='first')
              if cap_list else pd.DataFrame(columns=['run_id','cap_path','spans','seq_len']))
    
    conn.close()
    
    df = run_df.merge(dist_df, on='run_id', how='left')
    df = df.merge(cap_df, on='run_id', how='left')
    df['band'] = band_label
    # injection key: pools all user-task contexts for the same injection
    df['injection'] = df.apply(
        lambda r: f"{r['suite']}/{r['injection_task_id']}", axis=1)
    # full example key (suite/user_task/injection_task) kept for reference
    df['example'] = df.apply(
        lambda r: f"{r['suite']}/{r['user_task_id']}/{r['injection_task_id']}", axis=1)
    return df

In [ ]:
# dfs2 = []
# for band in BANDS:
#     df = load_log_dir(LOG2_ROOT / band, band_label=band)
#     dfs2.append(df)
# all2 = pd.concat(dfs2, ignore_index=True)

# flips2     = all2[all2['perturbation_type'] == 'flip'].copy().reset_index(drop=True)
# baselines2 = all2[all2['source'] == 'baseline'].copy().reset_index(drop=True)

# print(f'logs_2 flip runs:  {len(flips2)}')
# print(f'logs_2 baselines:  {len(baselines2)}')
# print(f'flip caps present: {flips2["cap_path"].notna().sum()}')
# print()
# pivot = (flips2.groupby(['example','band'])['success']
#          .agg(n='count', rate='mean').round(3).reset_index())
# print(pivot.to_string(index=False))

In [ ]:
unc_df        = load_log_dir(LOG_UNC, band_label='unconstrained')
flips_unc     = unc_df[unc_df['perturbation_type'] == 'flip'].copy().reset_index(drop=True)
baselines_unc = unc_df[unc_df['source'] == 'baseline'].copy().reset_index(drop=True)

# Aliases used throughout analysis cells
flips2     = flips_unc.copy()
baselines2 = baselines_unc.copy()

print(f'unconstrained flip runs:  {len(flips2)}')
print(f'  cap_path present:       {flips2["cap_path"].notna().sum()}')
print(f'  fluency non-null/non-zero: {(flips2["fluency"] > 0).sum()}')
print()

# Per-injection summary (pooled across user-task contexts)
inj_summary = (flips2.groupby(['injection', 'perturbation_N'])['success']
               .agg(n='count', success_rate='mean').round(3).reset_index())
print('Per-injection success rate by N:')
print(inj_summary.to_string(index=False))
print()

# Context coverage
ctx_counts = unc_df.groupby(['injection', 'user_task_id']).size().reset_index(name='runs')
print(f'Contexts per injection:')
for inj, grp in ctx_counts.groupby('injection'):
    print(f'  {inj}: {len(grp)} user-task contexts')

In [ ]:
# Context success rate heatmap — baseline success per (injection × user_task_id)
baseline_success = (baselines2.groupby(['injection', 'user_task_id'])['success']
                   .mean().reset_index(name='baseline_success_rate'))

injections = sorted(baseline_success['injection'].unique())
user_tasks = sorted(baseline_success['user_task_id'].unique())

pivot = baseline_success.pivot(index='injection', columns='user_task_id',
                               values='baseline_success_rate')
pivot = pivot.reindex(index=injections, columns=user_tasks)

fig, ax = plt.subplots(figsize=(max(8, len(user_tasks) * 0.7), len(injections) + 1))
im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1,
               interpolation='nearest')
plt.colorbar(im, ax=ax, label='Baseline success rate')
ax.set_xticks(range(len(user_tasks)))
ax.set_xticklabels(user_tasks, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(injections)))
ax.set_yticklabels(injections, fontsize=8)
ax.set_title('Baseline Injection Success Rate by Context\n(injection × user_task_id)')

for i in range(len(injections)):
    for j in range(len(user_tasks)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.0%}', ha='center', va='center',
                    fontsize=7, color='black' if 0.3 < val < 0.7 else 'white')

plt.tight_layout()
plt.show()

print('\nBaseline runs per context:')
ctx_baseline = baselines2.groupby(['injection', 'user_task_id']).size().reset_index(name='n')
print(ctx_baseline.to_string(index=False))

---
## 2. Sample Coverage — How Much Can We Sample?

Left block: `profiling_logs_2` — constrained flips, 3 examples, per band×N.  
Right: `profiling_logs_unconstrained` — continuous distance (banking only).

*Takeaway*: distance and N are tightly coupled. High-distance requires high N and produces fewer samples that pass distance filter.

In [ ]:
# fig = plt.figure(figsize=(18, 5))
# # Left 3 subplots: one per example, bar chart of counts per band/N
# for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
#     suite, ut, it = ex_tuple
#     ex_str = f'{suite}/{ut}/{it}'
#     ax = fig.add_subplot(1, 4, col + 1)
    
#     ex_data = flips2[flips2['example'] == ex_str]
#     pivot = ex_data.groupby(['band', 'perturbation_N']).size().reset_index(name='count')
    
#     band_list = BANDS
#     n_vals = sorted(pivot['perturbation_N'].unique())
#     width = 0.15
#     x = np.arange(len(n_vals))
    
#     for bi, band in enumerate(band_list):
#         sub = pivot[pivot['band'] == band]
#         heights = [sub[sub['perturbation_N'] == n]['count'].values[0]
#                    if len(sub[sub['perturbation_N'] == n]) else 0 for n in n_vals]
#         ax.bar(x + bi * width, heights, width, label=band,
#                color=BAND_COLORS[band], alpha=0.85)
    
#     ax.set_title(ex_label)
#     ax.set_xlabel('N (tokens flipped)')
#     ax.set_ylabel('# runs')
#     ax.set_xticks(x + width * 2)
#     ax.set_xticklabels([int(n) for n in n_vals])
#     if col == 0:
#         ax.legend(title='distance band', fontsize=7)

# # Right subplot: unconstrained — N vs fluency distance
# ax4 = fig.add_subplot(1, 4, 4)
# for n_val, grp in flips_unc.groupby('perturbation_N'):
#     ax4.scatter([n_val]*len(grp), grp['fluency'], alpha=0.2, s=6, color='#3498db')
# ax4.set_xlabel('N (tokens flipped)')
# ax4.set_ylabel('Fluency distance')
# ax4.set_title('Unconstrained: N vs fluency distance\n(banking only)')
# # Overlay mean
# unc_means = flips_unc.groupby('perturbation_N')['fluency'].mean()
# ax4.plot(unc_means.index, unc_means.values, 'k-o', linewidth=2, markersize=4, label='mean')
# ax4.legend()

# plt.suptitle('Sample Coverage', fontsize=12, y=1.01)
# plt.tight_layout()
# plt.show()

---
## 3. Success Rate vs N and Fluency Distance

Top row: constrained experiments — 3 examples side by side.  
Bottom: unconstrained banking example with continuous distance axis.

Higher N → more tokens flipped → higher fluency distance → lower success rate.  
The distance band is a proxy for perturbation severity.

In [ ]:
# fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)

# for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
#     suite, ut, it = ex_tuple
#     ex_str = f'{suite}/{ut}/{it}'
#     ax = axes[col]
#     ex_data = flips2[flips2['example'] == ex_str]
    
#     for band in BANDS:
#         sub = ex_data[ex_data['band'] == band]
#         if sub.empty: continue
#         grouped = sub.groupby('perturbation_N')['success'].agg(rate='mean', n='count').reset_index()
#         ax.plot(grouped['perturbation_N'], grouped['rate'],
#                 'o-', color=BAND_COLORS[band], label=f'{band}', linewidth=1.5, markersize=4)
    
#     ax.set_title(ex_label)
#     ax.set_xlabel('N (tokens flipped)')
#     ax.set_xlim(left=0)
#     ax.set_ylim(-0.05, 1.05)
#     ax.axhline(0.5, color='grey', linestyle=':', linewidth=1)
#     if col == 0:
#         ax.set_ylabel('Success rate')
#         ax.legend(title='distance band', ncol=2, fontsize=7)

# plt.suptitle('Success Rate vs N — Constrained Experiments (flips only)', fontsize=11)
# plt.tight_layout()
# plt.show()

---
## 4. Unconstrained Banking Example — Continuous Distance View

The unconstrained run has no attention captures, but provides a clean view of how  
fluency distance (continuous) relates to success rate for a wide range of N values.  
This anchors the distance axis for the constrained analysis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: success rate vs N
ax = axes[0]
grouped = flips_unc.groupby('perturbation_N')['success'].agg(rate='mean', n='count').reset_index()
ax.bar(grouped['perturbation_N'].astype(str), grouped['rate'],
       color='#3498db', alpha=0.8, edgecolor='white')
for _, r in grouped.iterrows():
    ax.text(str(int(r['perturbation_N'])), r['rate'] + 0.005, f"n={r['n']}",
            ha='center', fontsize=7, color='#555')
ax.set_xlabel('N (tokens flipped)')
ax.set_ylabel('Success rate')
ax.set_title('Unconstrained banking: success rate vs N')
ax.set_ylim(0, 0.25)

# Right: success vs fluency distance (binned)
ax = axes[1]
bins = np.arange(0, 11, 0.5)
flips_unc['dist_bin'] = pd.cut(flips_unc['fluency'], bins=bins, labels=bins[:-1]+0.25)
bin_stats = (flips_unc.groupby('dist_bin', observed=True)['success']
             .agg(rate='mean', n='count').reset_index())
bin_stats['dist_bin'] = bin_stats['dist_bin'].astype(float)
ax.scatter(bin_stats['dist_bin'], bin_stats['rate'],
           s=bin_stats['n'] * 0.5, alpha=0.7, color='#e74c3c', edgecolors='white', linewidth=0.5)
ax.plot(bin_stats['dist_bin'], bin_stats['rate'], color='#e74c3c', alpha=0.5, linewidth=1)
ax.set_xlabel('Fluency distance (binned, 0.5-wide)')
ax.set_ylabel('Success rate')
ax.set_title('Success rate vs fluency distance\n(point size ∝ sample count)')

plt.suptitle('Unconstrained Banking — Distance and Success', fontsize=11)
plt.tight_layout()
plt.show()

---
## 5. Precompute Attention Summaries

For each captured run, load the attention `.pt` file and compute:
- `mean_heads[l, g, s]` — payload/tag attention mass, averaged over 64 heads,  
  for each layer `l` ∈ {0..23}, tag group `g`, reasoning step `s` ∈ {0..9}
- `per_head[g, h, s]` — per-head attention mass averaged over layers

Runs are then tagged as `success / failure / baseline` for downstream analysis.

**Note on architecture**: GPT-OSS alternates local (~128-token sliding window) and  
global (full context) attention. Even layers = local, odd = global.  
The attack payload is ≤ 70 tokens before the reasoning boundary, so it's  
visible in **both** local and global layers. User/developer prompt regions  
are only visible in global layers.

In [ ]:
def compute_cap_summary(cap_path, spans, seq_len):
    """Compute per-tag attention mass summary from a cap file.
    Returns dict with 'mean_heads' [N_LAYERS, N_GRPS, N_STEPS]
                  and 'per_head'   [N_GRPS, N_HEADS, N_STEPS]
    """
    full_path = cap_path if os.path.exists(cap_path) else os.path.join('profiler', cap_path)
    if not os.path.exists(full_path):
        return None
    try:
        cap = torch.load(full_path, map_location='cpu', weights_only=False)
    except Exception:
        return None
    
    attn_dict = cap['attention']
    actual_len = attn_dict[0].shape[-1]
    actual_steps = attn_dict[0].shape[1]
    
    group_masks = np.zeros((N_GRPS, actual_len), dtype=bool)
    for gi, (grp, tags) in enumerate(TAG_GROUPS.items()):
        for sp in spans:
            if sp['tag'] in tags:
                s, e = sp['start'], min(sp['end'], actual_len)
                if s < e:
                    group_masks[gi, s:e] = True
    
    steps = min(actual_steps, N_STEPS)
    mean_heads = np.zeros((N_LAYERS, N_GRPS, N_STEPS), dtype=np.float32)
    per_head   = np.zeros((N_GRPS, N_HEADS, N_STEPS), dtype=np.float32)
    
    for layer_idx, attn_layer in attn_dict.items():
        a = attn_layer.float().numpy()  # [N_HEADS, actual_steps, actual_len]
        for gi in range(N_GRPS):
            idxs = np.where(group_masks[gi])[0]
            if len(idxs) == 0:
                continue
            mass = a[:, :steps, :][:, :, idxs].sum(-1)  # [N_HEADS, steps]
            mean_heads[layer_idx, gi, :steps] = mass.mean(0)
            per_head[gi, :, :steps]           += mass
    
    per_head /= N_LAYERS
    return {'mean_heads': mean_heads, 'per_head': per_head}


# Gather all rows that need caps: flips2 with cap_path + baselines2 with cap_path
base_sub = baselines2[baselines2['cap_path'].notna()].copy()
base_sub['band'] = 'baseline'

to_load = pd.concat([
    flips2[flips2['cap_path'].notna()][['run_id','cap_path','spans','seq_len','success','injection','band']],
    base_sub[['run_id','cap_path','spans','seq_len','success','injection','band']],
], ignore_index=True)

print(f'Loading {len(to_load)} cap files...')
cap_summaries = {}  # run_id -> summary dict
failed = 0

for _, row in tqdm(to_load.iterrows(), total=len(to_load), desc='caps'):
    spans = row['spans']
    if not isinstance(spans, list):
        failed += 1; continue
    summary = compute_cap_summary(row['cap_path'], spans, row['seq_len'])
    if summary is None:
        failed += 1; continue
    cap_summaries[row['run_id']] = summary

print(f'Loaded {len(cap_summaries)} summaries ({failed} failed)')

In [ ]:
flips2['has_cap'] = flips2['run_id'].isin(cap_summaries)
baselines2['has_cap'] = baselines2['run_id'].isin(cap_summaries)

def condition(row):
    if row.get('source') == 'baseline' or row.get('perturbation_type') == 'none':
        return 'baseline'
    return 'success' if row['success'] else 'failure'

flips2['condition'] = flips2.apply(condition, axis=1)
baselines2['condition'] = 'baseline'

# Per injection counts (pooled across all user-task contexts)
cond_counts = (flips2[flips2['has_cap']]
               .groupby(['injection', 'condition']).size().reset_index(name='n'))
print('Flip cap counts per injection (pooled across contexts):')
print(cond_counts.to_string(index=False))
print()
print('Per-context breakdown:')
ctx_cond = (flips2[flips2['has_cap']]
            .groupby(['injection', 'user_task_id', 'condition']).size().reset_index(name='n'))
print(ctx_cond.to_string(index=False))

---
## 6. Architecture Checkpoint — Local vs Global Layers

GPT-OSS uses alternating local/global attention. Before analysing attention by layer,  
confirm which layers can see which parts of the sequence.  
This matters: **developer/user prompt is only visible in global (odd) layers**;  
the **attack payload is visible in both** (it's ≤ 70 tokens before the boundary).

In [ ]:
# Take one baseline cap and measure effective attention window per layer
base_row = baselines2[baselines2['cap_path'].notna()].iloc[0]
cap_path = base_row['cap_path']
full_path = cap_path if os.path.exists(cap_path) else os.path.join('profiler', cap_path)
cap = torch.load(full_path, map_location='cpu', weights_only=False)
attn_dict = cap['attention']
seq_len   = cap['seq_len']
spans     = base_row['spans']

# For step 0 (first reasoning token), find first nonzero attention position
# across all 64 heads (max over heads = most permissive window)
first_visible = []
for l in range(N_LAYERS):
    a = attn_dict[l].float().numpy()  # [64, 10, seq_len]
    max_attn = a[:, 0, :].max(0)      # [seq_len] — max over heads at step 0
    nonzero  = np.where(max_attn > 1e-6)[0]
    first_visible.append(nonzero[0] if len(nonzero) else seq_len)

# Build tag group coverage map
tag_ranges = {}
for grp, tags in TAG_GROUPS.items():
    spans_for_grp = [sp for sp in spans if sp['tag'] in tags]
    if spans_for_grp:
        tag_ranges[grp] = (min(sp['start'] for sp in spans_for_grp),
                           max(sp['end']   for sp in spans_for_grp))

fig, ax = plt.subplots(figsize=(12, 4))
layers = np.arange(N_LAYERS)
ax.bar(layers, [seq_len - fv for fv in first_visible], bottom=first_visible,
       color=['#3498db' if l in GLOBAL_LAYERS else '#e74c3c' for l in layers],
       alpha=0.7, label='visible window')

# Mark tag group start positions as horizontal lines
grp_y_colors = {grp: GRP_COLORS[grp] for grp in tag_ranges}
for grp, (start, end) in tag_ranges.items():
    ax.axhline(start, color=GRP_COLORS[grp], linestyle='--', linewidth=1.2, alpha=0.8,
               label=f'{grp} start={start}')

ax.set_xlabel('Layer')
ax.set_ylabel('Token position')
ax.set_title(f'Effective attention window per layer (seq_len={seq_len}, query=first reasoning token)\n'
             'Blue bar = global layer, Red bar = local (sliding window). '
             'Dashed lines = start of each semantic group.')
ax.invert_yaxis()  # position 0 at top
ax.set_xticks(layers)
ax.legend(loc='lower right', fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

print('\nTag group positions:')
for grp, (s, e) in sorted(tag_ranges.items(), key=lambda x: x[1][0]):
    in_local = s > (seq_len - 130)
    print(f'  {grp:12s}: tokens {s:4d}–{e:4d}  ({"LOCAL-visible" if in_local else "global-only"})')

---
## 7. Attention Trajectory — Per Reasoning Step

For each reasoning step (0–9), how much attention lands on each semantic group?  
Averaged over all layers and all 64 heads.

**Row 1**: Payload attention only — baseline / success / failure for all 3 examples.  
**Row 2**: All semantic groups, success vs failure (global layers only — the ones that can see the prompt).

In [ ]:
def collect_trajectories(run_ids, layers=None):
    """Return array [n_runs, N_GRPS, N_STEPS] averaged over specified layers (all if None)."""
    arrs = []
    for rid in run_ids:
        if rid not in cap_summaries: continue
        mh = cap_summaries[rid]['mean_heads']  # [N_LAYERS, N_GRPS, N_STEPS]
        if layers is not None:
            mh = mh[layers]
        arrs.append(mh.mean(0))  # [N_GRPS, N_STEPS]
    return np.array(arrs) if arrs else None  # [n_runs, N_GRPS, N_STEPS]


steps = np.arange(N_STEPS)
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

gi_payload = GRP_NAMES.index('payload')

for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
    suite, inj_task = ex_tuple
    inj_str = f'{suite}/{inj_task}'

    base_ids    = baselines2[baselines2['injection'] == inj_str]['run_id'].tolist()
    success_ids = flips2[(flips2['injection'] == inj_str) & (flips2['success'])]['run_id'].tolist()
    failure_ids = flips2[(flips2['injection'] == inj_str) & (~flips2['success'])]['run_id'].tolist()

    # ROW 0: Payload attention only
    ax = axes[0, col]
    for ids, label, ls, alpha in [
        (base_ids,    'baseline', ':',  0.9),
        (success_ids, 'success',  '-',  0.9),
        (failure_ids, 'failure',  '--', 0.7),
    ]:
        arr = collect_trajectories(ids)
        if arr is None or len(arr) == 0: continue
        traj = arr[:, gi_payload, :]  # [n_runs, N_STEPS]
        mean = traj.mean(0)
        sem  = traj.std(0) / max(len(traj)**0.5, 1)
        ax.plot(steps, mean, ls=ls, color=GRP_COLORS['payload'], linewidth=2, label=f'{label} (n={len(arr)})')
        ax.fill_between(steps, mean - sem, mean + sem, alpha=0.15, color=GRP_COLORS['payload'])
    ax.set_title(f'{ex_label}\nPayload attention (all layers, all contexts pooled)')
    ax.set_xlabel('Reasoning step')
    ax.set_ylabel('Mean attention mass')
    ax.set_xticks(steps)
    ax.legend(fontsize=7)

    # ROW 1: All groups, success vs failure, global layers only
    ax = axes[1, col]
    global_layers_arr = np.array(GLOBAL_LAYERS)

    for ids, ls, alpha in [(success_ids, '-', 0.9), (failure_ids, '--', 0.7)]:
        arr = collect_trajectories(ids, layers=global_layers_arr)
        if arr is None or len(arr) == 0: continue
        for gi, grp in enumerate(GRP_NAMES):
            if grp in ('control', 'assistant'): continue
            traj = arr[:, gi, :]
            mean = traj.mean(0)
            label = f'{grp} ({"success" if ls == "-" else "failure"})' if col == 0 else ''
            ax.plot(steps, mean, ls=ls, color=GRP_COLORS[grp],
                    linewidth=1.5, alpha=alpha, label=label)

    ax.set_title(f'{ex_label}\nAll groups, global layers — solid=success dashed=failure')
    ax.set_xlabel('Reasoning step')
    ax.set_ylabel('Mean attention mass')
    ax.set_xticks(steps)
    if col == 0:
        ax.legend(fontsize=6, ncol=2)

plt.suptitle('Attention Trajectory per Reasoning Step (pooled across user-task contexts)', fontsize=12)
plt.tight_layout()
plt.show()

---
## 8. Where in the Network? Layer × Step Attention Heatmap

For the **payload** tag group: mean attention mass at each (layer, step) position,  
averaged over 64 heads and all available runs of each condition.

3 columns = 3 examples.  
3 rows = {success mean, failure mean, signed delta = success − failure}.

Local layers (even) can see the payload; global layers (odd) see the full context.  
Red = higher success attention to payload; Blue = higher failure attention.

In [ ]:
gi_payload = GRP_NAMES.index('payload')

def mean_heatmap(run_ids, gi=gi_payload):
    arrs = []
    for rid in run_ids:
        if rid not in cap_summaries: continue
        arrs.append(cap_summaries[rid]['mean_heads'][:, gi, :])  # [N_LAYERS, N_STEPS]
    return np.array(arrs).mean(0) if arrs else None  # [N_LAYERS, N_STEPS]


fig, axes = plt.subplots(3, 3, figsize=(18, 12))
row_titles = ['Success (mean)', 'Failure (mean)', 'Delta (success − failure)']
cmaps = ['Reds', 'Blues', 'RdBu_r']

delta_maps = []
hm_maps = {}

for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
    suite, inj_task = ex_tuple
    inj_str = f'{suite}/{inj_task}'

    success_ids = flips2[(flips2['injection'] == inj_str) & (flips2['success'])]['run_id'].tolist()
    failure_ids = flips2[(flips2['injection'] == inj_str) & (~flips2['success'])]['run_id'].tolist()

    hm_succ = mean_heatmap(success_ids)
    hm_fail = mean_heatmap(failure_ids)

    if hm_succ is None or hm_fail is None:
        continue

    hm_delta = hm_succ - hm_fail
    delta_maps.append(np.abs(hm_delta).max())
    hm_maps[col] = (hm_succ, hm_fail, hm_delta)

if not hm_maps:
    print("No heatmap data (need success AND failure caps). Run full precompute.")
    vmax_delta, vmax_abs = 0.01, 0.01
else:
    vmax_delta = max(delta_maps) if delta_maps else 0.01
    vmax_abs   = max(max(hm_maps[c][0].max(), hm_maps[c][1].max()) for c in hm_maps)

for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
    if col not in hm_maps: continue
    hm_succ, hm_fail, hm_delta = hm_maps[col]

    for row, (hm, cmap, row_title) in enumerate(zip(
            [hm_succ, hm_fail, hm_delta], cmaps, row_titles)):
        ax = axes[row, col]

        if row < 2:
            im = ax.imshow(hm, aspect='auto', cmap=cmap, vmin=0, vmax=vmax_abs,
                           interpolation='nearest')
        else:
            im = ax.imshow(hm, aspect='auto', cmap=cmap,
                           vmin=-vmax_delta, vmax=vmax_delta, interpolation='nearest')

        plt.colorbar(im, ax=ax, shrink=0.8)
        ax.set_xlabel('Reasoning step')
        ax.set_ylabel('Layer')
        ax.set_xticks(range(N_STEPS))
        ax.set_yticks(range(0, N_LAYERS, 2))
        ax.set_yticklabels(range(0, N_LAYERS, 2), fontsize=7)

        for l in LOCAL_LAYERS:
            ax.axhline(l - 0.5, color='white', linewidth=0.3, alpha=0.4)
        for l in LOCAL_LAYERS:
            ax.add_patch(plt.Rectangle((-0.5, l-0.5), N_STEPS, 1,
                                       fc='yellow', alpha=0.06, zorder=0))

        ax.set_title(f'{ex_label}\n{row_title}', fontsize=9)

plt.suptitle('Payload Attention Mass [Layer × Step] — pooled across contexts\n'
             'Yellow bands = local (sliding-window) layers, white = global layers',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

---
## 9. Which Heads Carry the Signal?

At the most discriminative layer (highest |success_mean − failure_mean| for payload),  
show the per-head payload attention for every one of the 64 heads.

Left: box plots per head, colored by condition (success/failure).  
Right: head-wise mean difference (success − failure), sorted by magnitude.  
Identifies the specific heads most sensitive to injection state.

In [ ]:
gi_payload = GRP_NAMES.index('payload')

layer_deltas = np.zeros(N_LAYERS)
for col, (ex_tuple, _) in enumerate(zip(EXAMPLES, EX_LABELS)):
    suite, inj_task = ex_tuple
    inj_str = f'{suite}/{inj_task}'
    success_ids = flips2[(flips2['injection'] == inj_str) & (flips2['success'])]['run_id'].tolist()
    failure_ids = flips2[(flips2['injection'] == inj_str) & (~flips2['success'])]['run_id'].tolist()
    for ids_list, sign in [(success_ids, 1), (failure_ids, -1)]:
        for rid in ids_list:
            if rid not in cap_summaries: continue
            layer_deltas += sign * cap_summaries[rid]['mean_heads'][:, gi_payload, :].mean(-1)

best_layer_global = GLOBAL_LAYERS[np.argmax(np.abs(layer_deltas[GLOBAL_LAYERS]))]
best_layer_local  = LOCAL_LAYERS [np.argmax(np.abs(layer_deltas[LOCAL_LAYERS ]))]
print(f'Most discriminative global layer: {best_layer_global}  (delta={layer_deltas[best_layer_global]:.4f})')
print(f'Most discriminative local  layer: {best_layer_local}   (delta={layer_deltas[best_layer_local]:.4f})')
FOCUS_LAYER = best_layer_global

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
    suite, inj_task = ex_tuple
    inj_str = f'{suite}/{inj_task}'
    ax = axes[col]

    succ_per_head = []
    fail_per_head = []

    for condition_ids, store in [
        (flips2[(flips2['injection']==inj_str)&(flips2['success'])]['run_id'], succ_per_head),
        (flips2[(flips2['injection']==inj_str)&(~flips2['success'])]['run_id'], fail_per_head),
    ]:
        for rid in condition_ids:
            if rid not in cap_summaries: continue
            ph = cap_summaries[rid]['per_head'][gi_payload, :, :]  # [N_HEADS, N_STEPS]
            store.append(ph.mean(-1))  # [N_HEADS]

    if not succ_per_head or not fail_per_head:
        ax.set_title(f'{ex_label}\n(no data)')
        continue

    succ_arr = np.array(succ_per_head)  # [n_succ, N_HEADS]
    fail_arr = np.array(fail_per_head)  # [n_fail, N_HEADS]
    head_delta = succ_arr.mean(0) - fail_arr.mean(0)

    ax.bar(range(N_HEADS), succ_arr.mean(0), alpha=0.6, color='#2ecc71', label='success mean')
    ax.bar(range(N_HEADS), fail_arr.mean(0), alpha=0.6, color='#e74c3c', label='failure mean')

    ax2 = ax.twinx()
    ax2.step(range(N_HEADS), head_delta, color='black', linewidth=0.8, alpha=0.6)
    ax2.axhline(0, color='black', linewidth=0.5, linestyle=':')
    ax2.set_ylabel('Success − failure delta', fontsize=8)

    ax.set_xlabel('Head index')
    ax.set_ylabel('Mean payload attention mass')
    ax.set_title(f'{ex_label}\nPer-head payload attention (layer-avg, all contexts pooled)')
    if col == 0:
        ax.legend(fontsize=7)

plt.suptitle('Per-Head Payload Attention — All 64 Heads\n(averaged over layers and reasoning steps)',
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Top-10 most discriminative heads (by |success_mean - failure_mean|), pooled across injections
all_deltas = np.zeros(N_HEADS)
for col, (ex_tuple, _) in enumerate(zip(EXAMPLES, EX_LABELS)):
    suite, inj_task = ex_tuple
    inj_str = f'{suite}/{inj_task}'
    succ_ph = [cap_summaries[rid]['per_head'][gi_payload].mean(-1)
               for rid in flips2[(flips2['injection']==inj_str)&(flips2['success'])]['run_id']
               if rid in cap_summaries]
    fail_ph = [cap_summaries[rid]['per_head'][gi_payload].mean(-1)
               for rid in flips2[(flips2['injection']==inj_str)&(~flips2['success'])]['run_id']
               if rid in cap_summaries]
    if succ_ph and fail_ph:
        all_deltas += np.array(succ_ph).mean(0) - np.array(fail_ph).mean(0)

top_heads = np.argsort(np.abs(all_deltas))[::-1][:10]
print('Top-10 most discriminative heads (pooled across 3 injections, all contexts):')
for h in top_heads:
    print(f'  head {h:3d}: delta = {all_deltas[h]:+.5f}')

---
## 10. Payload Attention Mass vs Fluency Distance

Using the actual fluency distance values from the profiling logs.  
x-axis: continuous fluency distance.  
y-axis: mean payload attention (averaged over layers, heads, reasoning steps).  
Coloured by injection success.

*If attention tracks injection state*, we'd expect successful runs to cluster at  
higher payload attention regardless of distance, while failed runs would have lower  
payload attention — providing a detector signal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

gi_payload = GRP_NAMES.index('payload')

for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
    suite, inj_task = ex_tuple
    inj_str = f'{suite}/{inj_task}'
    ax = axes[col]

    ex_flips = flips2[(flips2['injection'] == inj_str) & flips2['has_cap']].copy()

    if ex_flips.empty:
        continue

    attn_vals = []
    for _, row in ex_flips.iterrows():
        rid = row['run_id']
        if rid not in cap_summaries: continue
        mh = cap_summaries[rid]['mean_heads']  # [N_LAYERS, N_GRPS, N_STEPS]
        mean_payload = mh[:, gi_payload, :].mean()
        attn_vals.append({
            'run_id':  rid,
            'fluency': row['fluency'],
            'success': row['success'],
            'N':       row['perturbation_N'],
            'user_task': row['user_task_id'],
            'payload_attn': mean_payload,
        })

    if not attn_vals:
        continue

    av = pd.DataFrame(attn_vals)

    for succ, color, label, zorder in [
        (True,  '#2ecc71', 'success', 3),
        (False, '#e74c3c', 'failure', 2),
    ]:
        sub = av[av['success'] == succ]
        ax.scatter(sub['fluency'], sub['payload_attn'],
                   c=color, alpha=0.4, s=20, label=f'{label} (n={len(sub)})', zorder=zorder)
        if len(sub) >= 4:
            x, y = sub['fluency'].values, sub['payload_attn'].values
            z = np.polyfit(x, y, 1)
            xr = np.linspace(x.min(), x.max(), 50)
            ax.plot(xr, np.polyval(z, xr), color=color, linewidth=2, zorder=4)

    ax.set_xlabel('Fluency distance')
    ax.set_ylabel('Mean payload attention mass')
    ax.set_title(f'{ex_label}\n(all contexts pooled)')
    ax.legend(fontsize=7)

plt.suptitle('Payload Attention vs Fluency Distance (all layers & heads averaged, all contexts)',
             fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

for row_i, (layer_set, layer_label) in enumerate([
    (np.array(GLOBAL_LAYERS), 'Global layers (full-context, odd)'),
    (np.array(LOCAL_LAYERS),  'Local layers (sliding-window ~128, even)'),
]):
    for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
        suite, inj_task = ex_tuple
        inj_str = f'{suite}/{inj_task}'
        ax = axes[row_i, col]

        ex_flips = flips2[(flips2['injection'] == inj_str) & flips2['has_cap']]
        attn_vals = []
        for _, row in ex_flips.iterrows():
            rid = row['run_id']
            if rid not in cap_summaries: continue
            mh = cap_summaries[rid]['mean_heads']
            mean_payload = mh[layer_set][:, gi_payload, :].mean()
            attn_vals.append({'fluency': row['fluency'], 'success': row['success'],
                              'payload_attn': mean_payload})
        if not attn_vals: continue
        av = pd.DataFrame(attn_vals)

        for succ, color, label in [
            (True, '#2ecc71', 'success'), (False, '#e74c3c', 'failure')]:
            sub = av[av['success'] == succ]
            ax.scatter(sub['fluency'], sub['payload_attn'],
                       c=color, alpha=0.4, s=15, label=f'{label} (n={len(sub)})')
            if len(sub) >= 4:
                x, y = sub['fluency'].values, sub['payload_attn'].values
                z = np.polyfit(x, y, 1)
                xr = np.linspace(x.min(), x.max(), 50)
                ax.plot(xr, np.polyval(z, xr), color=color, linewidth=2)

        ax.set_xlabel('Fluency distance')
        ax.set_ylabel('Mean payload attn')
        ax.set_title(f'{ex_label}\n{layer_label}', fontsize=8)
        if col == 0:
            ax.legend(fontsize=7)

plt.suptitle('Payload Attention vs Fluency Distance — Local vs Global Layers (all contexts)', fontsize=11)
plt.tight_layout()
plt.show()

---
## 11. Layer-wise Discriminability

For each layer, compute the **area under the ROC curve** (AUROC) for predicting injection  
success from payload attention mass (averaged over heads and reasoning steps).

AUROC = 0.5 → random, 1.0 → perfect.  
This identifies which layers carry the most information about injection state.

In [ ]:
def auroc(scores, labels):
    labels = np.asarray(labels, dtype=float)
    scores = np.asarray(scores, dtype=float)
    n_pos = labels.sum()
    n_neg = len(labels) - n_pos
    if n_pos == 0 or n_neg == 0:
        return 0.5
    order = np.argsort(-scores)
    tp = np.cumsum(labels[order])
    fp = np.cumsum(1 - labels[order])
    tpr = np.concatenate([[0], tp / n_pos])
    fpr = np.concatenate([[0], fp / n_neg])
    return float(np.trapz(tpr, fpr))


fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
    suite, inj_task = ex_tuple
    inj_str = f'{suite}/{inj_task}'
    ax = axes[col]

    ex_flips = flips2[(flips2['injection'] == inj_str) & flips2['has_cap']]

    run_data = []
    for _, row in ex_flips.iterrows():
        rid = row['run_id']
        if rid not in cap_summaries: continue
        mh = cap_summaries[rid]['mean_heads']  # [N_LAYERS, N_GRPS, N_STEPS]
        per_layer_score = mh[:, gi_payload, :].mean(-1)  # [N_LAYERS]
        run_data.append((int(row['success']), per_layer_score))

    if not run_data: continue

    labels = np.array([d[0] for d in run_data])
    scores_matrix = np.array([d[1] for d in run_data])  # [n_runs, N_LAYERS]

    aucs = [auroc(scores_matrix[:, l], labels) for l in range(N_LAYERS)]

    colors = ['#3498db' if l in GLOBAL_LAYERS else '#e74c3c' for l in range(N_LAYERS)]
    ax.bar(range(N_LAYERS), aucs, color=colors, alpha=0.8, edgecolor='white')
    ax.axhline(0.5, color='black', linewidth=1, linestyle=':', label='chance')
    ax.axhline(0.7, color='grey',  linewidth=0.8, linestyle='--', alpha=0.5)

    best_l = int(np.argmax(aucs))
    ax.annotate(f'L{best_l}\n{aucs[best_l]:.2f}',
                xy=(best_l, aucs[best_l]), xytext=(best_l + 1.5, aucs[best_l] + 0.02),
                fontsize=7, arrowprops=dict(arrowstyle='->', lw=0.8))

    ax.set_xlabel('Layer')
    ax.set_ylabel('AUROC')
    ax.set_title(f'{ex_label}\nPayload-attention AUROC per layer (all contexts pooled)')
    ax.set_ylim(0.3, 1.0)
    ax.set_xticks(range(0, N_LAYERS, 2))

    legend_handles = [
        Patch(facecolor='#3498db', alpha=0.8, label='global layer'),
        Patch(facecolor='#e74c3c', alpha=0.8, label='local layer'),
        Line2D([0], [0], color='black', linestyle=':', linewidth=1, label='chance (0.5)'),
    ]
    if col == 0:
        ax.legend(handles=legend_handles, fontsize=7)

plt.suptitle('Layer-wise AUROC: Can Payload Attention Predict Injection Success?', fontsize=11)
plt.tight_layout()
plt.show()

---
## 12. Multi-tag AUROC — Which Semantic Region Is Most Informative?

Same AUROC analysis but for every semantic group, at the best layer.  
Shows whether other groups (e.g., developer, user, tool_env) also shift  
in attention as a function of injection success — and in which direction.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
    suite, inj_task = ex_tuple
    inj_str = f'{suite}/{inj_task}'
    ax = axes[col]

    ex_flips = flips2[(flips2['injection'] == inj_str) & flips2['has_cap']]
    run_data = []
    for _, row in ex_flips.iterrows():
        rid = row['run_id']
        if rid not in cap_summaries: continue
        mh = cap_summaries[rid]['mean_heads']  # [N_LAYERS, N_GRPS, N_STEPS]
        per_grp = mh[np.array(GLOBAL_LAYERS)][:, :, :].mean(axis=(0, 2))  # [N_GRPS]
        run_data.append((int(row['success']), per_grp))

    if not run_data: continue
    labels = np.array([d[0] for d in run_data])
    scores_matrix = np.array([d[1] for d in run_data])  # [n_runs, N_GRPS]

    aucs = [auroc(scores_matrix[:, gi], labels) for gi in range(N_GRPS)]

    bar_colors = [GRP_COLORS[grp] for grp in GRP_NAMES]
    ax.bar(range(N_GRPS), aucs, color=bar_colors, alpha=0.85, edgecolor='white')
    ax.axhline(0.5, color='black', linewidth=1, linestyle=':', label='chance')
    ax.set_xticks(range(N_GRPS))
    ax.set_xticklabels(GRP_NAMES, rotation=35, ha='right')
    ax.set_ylabel('AUROC')
    ax.set_title(f'{ex_label}\nAUROC per semantic group (global layers, all contexts)')
    ax.set_ylim(0.3, 1.0)
    ax.axhline(0.7, color='grey', linewidth=0.8, linestyle='--', alpha=0.5)

plt.suptitle('Per-Group AUROC at Global Layers — Which Semantic Region Predicts Success?',
             fontsize=11)
plt.tight_layout()
plt.show()

---
## 13. First Reasoning Token Only — Step 0

The first reasoning token is the most direct reflection of prompt processing  
(subsequent tokens attend heavily to prior reasoning tokens, diluting the prompt signal).  
Repeat the AUROC and attention-vs-distance plots using **only step 0**.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

for col, (ex_tuple, ex_label) in enumerate(zip(EXAMPLES, EX_LABELS)):
    suite, inj_task = ex_tuple
    inj_str = f'{suite}/{inj_task}'

    ex_flips = flips2[(flips2['injection'] == inj_str) & flips2['has_cap']]
    run_data = []
    for _, row in ex_flips.iterrows():
        rid = row['run_id']
        if rid not in cap_summaries: continue
        mh = cap_summaries[rid]['mean_heads']  # [N_LAYERS, N_GRPS, N_STEPS]
        step0_all    = mh[:, gi_payload, 0]
        step0_global = mh[np.array(GLOBAL_LAYERS), gi_payload, 0]
        run_data.append({
            'label':    int(row['success']),
            'fluency':  row['fluency'],
            'all_l':    step0_all.mean(),
            'global_l': step0_global.mean(),
            'per_layer': step0_all,
        })
    if not run_data: continue

    df_rd = pd.DataFrame(run_data)
    labels = df_rd['label'].values
    per_layer_mat = np.vstack(df_rd['per_layer'].values)  # [n_runs, N_LAYERS]

    # Row 0: AUROC per layer at step 0
    ax = axes[0, col]
    aucs_s0 = [auroc(per_layer_mat[:, l], labels) for l in range(N_LAYERS)]
    colors = ['#3498db' if l in GLOBAL_LAYERS else '#e74c3c' for l in range(N_LAYERS)]
    ax.bar(range(N_LAYERS), aucs_s0, color=colors, alpha=0.8, edgecolor='white')
    ax.axhline(0.5, color='black', linewidth=1, linestyle=':')
    ax.set_xlabel('Layer')
    ax.set_ylabel('AUROC')
    ax.set_title(f'{ex_label}\nAUROC per layer (step 0 only, all contexts)')
    ax.set_ylim(0.3, 1.0)
    ax.set_xticks(range(0, N_LAYERS, 2))

    # Row 1: Scatter — payload attn (step 0, all layers) vs fluency distance
    ax = axes[1, col]
    for succ, color, label in [(1, '#2ecc71', 'success'), (0, '#e74c3c', 'failure')]:
        sub = df_rd[df_rd['label'] == succ]
        ax.scatter(sub['fluency'], sub['all_l'], c=color, alpha=0.4, s=15,
                   label=f'{label} (n={len(sub)})')
        if len(sub) >= 4:
            x, y = sub['fluency'].values, sub['all_l'].values
            z = np.polyfit(x, y, 1)
            xr = np.linspace(x.min(), x.max(), 50)
            ax.plot(xr, np.polyval(z, xr), color=color, linewidth=2)
    ax.set_xlabel('Fluency distance')
    ax.set_ylabel('Payload attn (step 0)')
    ax.set_title(f'{ex_label}\nPayload attn at step 0 vs distance')
    if col == 0:
        ax.legend(fontsize=7)

plt.suptitle('Step 0 Analysis: First Reasoning Token Is the Cleanest Signal (all contexts)', fontsize=11)
plt.tight_layout()
plt.show()

---
## Summary

| Analysis | Key finding |
|---|---|
| **Sample coverage** | Fluency distance scales linearly with N; high-distance bands are sparsely populated. |
| **Success rate** | Injection success falls sharply with N (and thus distance) across all 3 examples. |
| **Architecture** | GPT-OSS alternates local (~128-token window) and global layers. Payload is visible in both; user/developer prompt only in global layers. |
| **Attention trajectory** | Payload attention is highest at step 0 and decays; successive reasoning tokens attend more to prior reasoning tokens. |
| **Layer × step heatmap** | The attention signal (success vs failure) concentrates in specific layers — inspect the delta heatmap to identify them. |
| **Per-head analysis** | A small subset of heads (out of 64) dominate the discriminative signal; these are candidates for mechanistic causal analysis. |
| **Attention vs distance** | Check whether payload attention tracks fluency distance (confound) or injection success (signal) independently. |
| **AUROC** | The layer-wise AUROC chart shows how readable the injection state is at each layer — the best layers are the extraction targets for classifiers. |

**Next steps**: fit a lightweight logistic classifier on the top-K head activations from the best layer to formally quantify injection detectability.